# Spectral Decomposition Approach


Simplifed noteebook demonstrating night sky decomposition approach.

Sky spectrum decomposed into model of airglow sky lines and continuum components:

I. Continuum components include:

- Moon (high-resolution solar spectrum rebinned to LVM sampling and convolved to LVM Gaussian LSF and multiplied by B-spline multiplicative continuum). This mimicks Moon itself and Zodi components.

- Diffuse components (interpolated from low-resolution PALACE diffuse continuum components)

  - Hydroperoxyl ($HO_2$): This is the predominant continuum component in the near-infrared range, characterized by a prominent emission peak at 1.51 µm.

  - Iron Monoxide ($FeO$) and other molecules: This component dominates the visual wavelength range (roughly 500 to 720 nm) and includes the $FeO$ "orange arc" bands, with potential additional contributions from $NiO$ or $OFeOH$.

  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).


II. Airglow sky components include:

  - Atomic Oxygen (O I) emission lines within the visual wavelength range.

  - Sodium (Na I): doublet at 5889.95 and 5895.92 Å, formed in the mesospheric Na layer at about 92 km by chemiluminescent reactions of meteoric sodium; typical D2 / D1 ≈ 1.7.
  
  - Potassium (K I): doublet at 7664.90 and 7698.96 Å, formed in the mesospheric K layer at about 89 km by chemistry similar to Na; typical D2 / D1 ≈ 1.67.
  
  - Nitrogen (N I): [ N I ] [NI] doublet at 5197.90 and 5200.26 Å, formed higher in the ionosphere at about 250 km via dissociative recombination; typical 5198 / 5200 ≈ 1.76.


  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).

  - Hydroxyl (OH): This component accounts for the hydroxyl emission lines in the visual wavelength range (roughly 500 to 720 nm).

# Manual in-place experiments -- ND

## Imports and helpers

Load the core packages and define the helper functions used throughout the notebook.

In [1]:
import plotly.graph_objects as go
from astropy.io import fits
from astropy.table import Table
import numpy as np

FACTOR = 1e14
LSF_SIGMA = 0.5
T_O2 = 191.5 # in K

PALACE_DIR = '../'
MEDIAN_STACK_DIR = '../'

f = MEDIAN_STACK_DIR+'lvmsframe_median_stack_1.2.1_limit100.fits'
fits.info(f)

wave = fits.getdata(f, "WAVE").astype(np.float64)
flx_sky1 = fits.getdata(f, "FLUX_SKY_NEAR").astype(np.float64) * FACTOR
flx_sky2 = fits.getdata(f, "FLUX_SKY_FAR").astype(np.float64) * FACTOR
flx_sci = fits.getdata(f, "FLUX_SCI").astype(np.float64) * FACTOR
meta = Table(fits.getdata(f, "META"))
flx_ivar = 1.0 + np.zeros_like(flx_sci)  # fits.getdata(f, "FLUX_IVAR").astype(np.float64)
# flx_err = 1.0 / np.sqrt(flx_ivar)
# flx_sci = flx_flux + flx_sky
flx_sci.shape


def plot_fit_result(result, idx):
    bestfit = result.bestfit
    bestfit_lsf = result.bestfit_lsf
    comp_Moon = result.components["moon"]
    comp_DIFFUSE = result.components["diffuse"]

    resid = flx_sci[idx] - bestfit
    resid_lsf = flx_sci[idx] - bestfit_lsf
    resid_level = -3.0 * np.nanstd(resid)

    err_plot = 1.0 / np.sqrt(np.where(flx_ivar[idx] > 0, flx_ivar[idx], np.nan))

    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=wave, y=flx_sci[idx], mode="lines", name="Observed",
                            line=dict(color="black", width=1)))
    fig.add_trace(go.Scattergl(x=wave, y=bestfit_lsf, mode="lines", name="Best-fit + LSF",
                            line=dict(color="darkorange", width=1.5)))
    fig.add_trace(go.Scattergl(x=wave, y=comp_Moon, mode="lines", name="Moon",
                            line=dict(color="#4e79a7", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=comp_DIFFUSE, mode="lines", name="Diffuse",
                            line=dict(color="#59a14f", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level + resid_lsf, mode="lines", name="Residual + LSF",
                            line=dict(color="seagreen", width=1)))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level + err_plot, mode="lines", name="+1sigma",
                            line=dict(color="gray", width=1, dash="dash")))
    fig.add_trace(go.Scattergl(x=wave, y=resid_level - err_plot, mode="lines", name="-1sigma",
                            line=dict(color="gray", width=1, dash="dash")))

    fig.update_layout(
        title={
            "text": (
                f"idx={idx} | T_O2={result.t_o2:.1f}±{result.t_o2_err:.1f} K<br>"
                f"{result.fit_summary}"
            ),
            "font": {"size": 14},
        },
        xaxis_title="λ (Å)",
        yaxis_title="Flux",
        template="plotly_white",
        height=520,
    )
    fig.show()


Filename: ../lvmsframe_median_stack_1.2.1_limit100.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      12   ()      
  1  WAVE          1 ImageHDU         9   (12401,)   float32   
  2  FLUX_SCI      1 ImageHDU        10   (12401, 100)   float32   
  3  FLUX_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  4  FLUX_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  5  FLUX_SCI_NOSKY    1 ImageHDU        10   (12401, 100)   float32   
  6  LSF_SCI       1 ImageHDU        10   (12401, 100)   float32   
  7  LSF_SKY_NEAR    1 ImageHDU        10   (12401, 100)   float32   
  8  LSF_SKY_FAR    1 ImageHDU        10   (12401, 100)   float32   
  9  META          1 BinTableHDU    107   100R x 48C   [512A, K, K, K, K, 32A, 32A, D, D, D, D, D, D, 8A, 8A, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, K, K, K, K, K, K, 16A, 512A]   


In [2]:
def thin_fits_every_n(input_path, output_path, n, row_hdu_name="META"):
    """Write a new FITS with every n-th row-like element kept.

    The function preserves HDU structure and headers. It identifies the row
    count from `row_hdu_name` (default: META), then slices any table HDU with
    that row count and any image HDU whose first axis matches that row count.
    """
    if n < 1:
        raise ValueError("n must be >= 1")

    with fits.open(input_path) as hdul:
        if row_hdu_name not in hdul:
            raise KeyError(f"HDU '{row_hdu_name}' not found in {input_path}")

        n_rows = len(hdul[row_hdu_name].data)
        keep = slice(None, None, n)

        out_hdus = []
        for hdu in hdul:
            header = hdu.header.copy()

            if isinstance(hdu, fits.PrimaryHDU):
                data = hdu.data
                if data is not None and getattr(data, "ndim", 0) >= 1 and data.shape[0] == n_rows:
                    data = data[keep, ...]
                out_hdus.append(fits.PrimaryHDU(data=data, header=header))

            elif isinstance(hdu, (fits.BinTableHDU, fits.TableHDU)):
                data = hdu.data
                if data is not None and len(data) == n_rows:
                    data = data[keep]
                out_hdus.append(type(hdu)(data=data, header=header, name=hdu.name))

            elif isinstance(hdu, (fits.ImageHDU, fits.CompImageHDU)):
                data = hdu.data
                if data is not None and getattr(data, "ndim", 0) >= 1 and data.shape[0] == n_rows:
                    data = data[keep, ...]
                out_hdus.append(type(hdu)(data=data, header=header, name=hdu.name))

            else:
                out_hdus.append(hdu.copy())

        fits.HDUList(out_hdus).writeto(output_path, overwrite=True)


# Example:
# thin_fits_every_n(
#     "../lvmsframe_median_stack_1.2.1_limit100.fits",
#     "../lvmsframe_median_stack_1.2.1_limit100_every5.fits",
#     n=5)

def results_to_fits(results, filename):
    """Write a list of SkyDecompResult objects to a FITS file.
    
    Scalar quantities go into a binary table (extension META).
    Spectral/coefficient arrays go into separate image extensions.
    
    Extensions:
        META         - BinTable with scalar fields per result
        COEF         - (n_results, n_coef) fit coefficients
        BESTFIT      - (n_results, n_wave) initial best-fit spectra
        BESTFIT_LSF  - (n_results, n_wave) LSF-refined best-fit spectra
        RESID        - (n_results, n_wave) residuals
        COMP_<KEY>   - (n_results, n_wave) per component spectra
    """
    rows = {
        "t_o2":               [r.t_o2 for r in results],
        "t_o2_err":           [r.t_o2_err for r in results],
        "reduced_chi2":       [r.reduced_chi2 for r in results],
        "r2":                 [r.r2 for r in results],
        "rms_resid":          [r.rms_resid for r in results],
        "resid_level":        [r.resid_level for r in results],
        "fit_status":         [r.fit_status for r in results],
        "fit_summary":        [r.fit_summary for r in results],
        "fit_elapsed_sec":    [r.fit_elapsed_sec for r in results],
        "peak_memory_mb":     [r.peak_memory_mb for r in results],
        "o2_fit_status":      [r.o2_fit_status for r in results],
        "o2_fit_summary":     [r.o2_fit_summary for r in results],
        "o2_fit_elapsed_sec": [r.o2_fit_elapsed_sec for r in results],
        "o2_valid_frac":      [r.o2_valid_frac for r in results],
    }
    t = Table(rows)

    def stack(attr):
        return np.vstack([getattr(r, attr) for r in results])

    # COEF header: store design_names as FITS keywords for reference
    coef_arr = stack("coef")
    coef_hdu = fits.ImageHDU(coef_arr, name="COEF")
    design_names = results[0].design_names
    for i, name in enumerate(design_names):
        coef_hdu.header[f"COEF{i:04d}"] = name

    hdul = fits.HDUList([
        fits.PrimaryHDU(),
        fits.BinTableHDU(t, name="META"),
        coef_hdu,
        fits.ImageHDU(stack("bestfit"),     name="BESTFIT"),
        fits.ImageHDU(stack("bestfit_lsf"), name="BESTFIT_LSF"),
        fits.ImageHDU(stack("resid"),       name="RESID"),
    ])

    comp_keys = list(results[0].components.keys())
    for key in comp_keys:
        arr = np.vstack([r.components[key] for r in results])
        hdul.append(fits.ImageHDU(arr, name=f"COMP_{key.upper()}"))

    hdul.writeto(filename, overwrite=True)
    print(f"Wrote {len(results)} results, {coef_arr.shape[1]} coefs, {len(comp_keys)} components → {filename}")

In [3]:
hdul = fits.open("lvmsframe_median_stack_1.2.1_every10_decomp_sci.fits")
#hdul = fits.open("../lvmsframe_median_stack_1.2.1_limit100.fits")
hdul.info()
t = Table(hdul["COEF"].data)
t.colnames[-20:]
# t = Table(hdul["META"].data)
# t.colnames
# t

Filename: lvmsframe_median_stack_1.2.1_every10_decomp_sci.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  META          1 BinTableHDU     37   1726R x 14C   [D, D, D, D, D, D, 6A, 116A, D, D, 1A, 115A, D, D]   
  2  COEF          1 BinTableHDU    893   1726R x 442C   [D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D,

['Moon_bs20',
 'Moon_bs21',
 'Moon_bs22',
 'Moon_bs23',
 'Moon_bs24',
 'Moon_bs25',
 'Moon_bs26',
 'Moon_bs27',
 'Moon_bs28',
 'HO2',
 'FeO',
 'O2Ac',
 'ATOM_K',
 'ATOM_N',
 'ATOM_Na',
 'ATOM_Og',
 'ATOM_Or',
 'ATOM_Orc_OI0777',
 'ATOM_Orc_OI0845',
 'O2_b01']

## VAE Modeling Experiments

In [4]:
# V1 baseline (stable): conditional VAE for decomposition coefficients
# This version models p(coef | context) with guarded training to avoid NaNs.
from astropy.io import fits
from astropy.table import Table
import numpy as np
import pandas
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.express as px
from torch.utils.data import TensorDataset, DataLoader

def _as_array(x):
    """Return numeric input as float32 array; skip non-numeric columns.

    This helper is used when reading FITS tables that can contain string/object
    metadata columns mixed with numeric fields.
    """
    arr = np.asarray(x)
    if arr.dtype.kind in ("U", "S", "O"):
        # Non-numeric fields are intentionally ignored in this baseline.
        return None
    return arr.astype(np.float32)

def _coerce_coef_hdu_to_table(coef_hdu):
    """Normalize COEF extension into an Astropy table.

    Supports both historical layouts:
    1) `BinTableHDU`/`TableHDU` with named columns.
    2) `ImageHDU` (2D array) where names may be stored in `COEFxxxx` header keys.
    """
    data = coef_hdu.data
    if isinstance(coef_hdu, (fits.BinTableHDU, fits.TableHDU)):
        return Table(data)
    # ImageHDU fallback: infer names from COEFxxxx header keywords if available.
    arr = np.asarray(data, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D COEF image, got shape={arr.shape}")
    n_coef = arr.shape[1]
    names = []
    for i in range(n_coef):
        key = f"COEF{i:04d}"
        names.append(str(coef_hdu.header.get(key, f"coef_{i:04d}")))
    return Table({name: arr[:, i] for i, name in enumerate(names)})

def _select_context_from_labels(meta, meta_upper, labels, base_name):
    """Build a single context vector using SKYE_/SKYW_ columns and row labels.

    For a requested base name (e.g., `moon_sep`), this function expects
    `SKYE_MOON_SEP` and `SKYW_MOON_SEP` columns. It then selects the right value
    per row according to label values (`SKYE` or `SKYW`).
    """
    e_key = f"SKYE_{base_name.upper()}"
    w_key = f"SKYW_{base_name.upper()}"
    if e_key not in meta_upper or w_key not in meta_upper:
        return None
    arr_e = _as_array(meta[meta_upper[e_key]])
    arr_w = _as_array(meta[meta_upper[w_key]])
    if arr_e is None or arr_w is None:
        raise ValueError(f"Labeled context columns for '{base_name}' are non-numeric.")
    is_e = labels == "SKYE"
    is_w = labels == "SKYW"
    if not np.all(is_e | is_w):
        bad = np.unique(labels[~(is_e | is_w)])
        raise ValueError(f"Unexpected label values: {bad}")
    out = np.where(is_e, arr_e, arr_w).astype(np.float32)
    return out

def read_decomp_dataset(
    decomp_fits_path,
    input_fits_path,
    context_columns,
    decomp_kind="sky1",
):
    """Read decomposition coefficients and aligned context from FITS inputs.

    Parameters
    ----------
    decomp_fits_path : str
        FITS file produced by decomposition (must contain COEF extension).
    input_fits_path : str
        Original input FITS file (must contain META extension).
    context_columns : list[str]
        Context feature names. If a name is not found directly in META, the
        function tries `skye_<name>`/`skyw_<name>` and chooses per-row values
        based on near/far sky labels.
    decomp_kind : str
        One of {"sky1", "sky2", "sci"}. Controls which label column is used.

    Returns
    -------
    coef_mat, ctx_mat, coef_names, ctx_names : tuple
        Filtered numeric matrices and the corresponding column-name lists.
    """
    if context_columns is None or len(context_columns) == 0:
        raise ValueError("context_columns must be a non-empty list of META column names.")
    kind = decomp_kind.lower()
    if kind not in ("sky1", "sky2", "sci"):
        raise ValueError("decomp_kind must be one of: 'sky1', 'sky2', 'sci'")

    # Load decomposition coefficients and input metadata.
    with fits.open(decomp_fits_path) as hdul_dec:
        coef_tbl = _coerce_coef_hdu_to_table(hdul_dec["COEF"])
    with fits.open(input_fits_path) as hdul_in:
        meta = Table(hdul_in["META"].data)

    # Build coefficient matrix in stable table-column order.
    coef_names = list(coef_tbl.colnames)
    coef_cols = []
    for name in coef_names:
        arr = _as_array(coef_tbl[name])
        if arr is not None:
            coef_cols.append(arr)
    if len(coef_cols) == 0:
        raise ValueError("No numeric coefficient columns found in COEF extension.")
    coef_mat = np.column_stack(coef_cols).astype(np.float32)

    # Build context matrix from explicit META columns and/or label-selected SKYE_/SKYW_.
    meta_upper = {c.upper(): c for c in meta.colnames}
    labels = None
    if kind in ("sky1", "sky2"):
        label_col = "SKY_NEAR_LABEL" if kind == "sky1" else "SKY_FAR_LABEL"
        if label_col not in meta_upper:
            raise KeyError(f"Missing required META label column: {label_col}")
        labels = np.char.upper(np.char.strip(np.asarray(meta[meta_upper[label_col]]).astype(str)))

    ctx_names = []
    ctx_cols = []
    missing_cols = []
    for cname in context_columns:
        key = cname.upper()
        if key in meta_upper:
            arr = _as_array(meta[meta_upper[key]])
            if arr is None:
                raise ValueError(f"Context column '{cname}' is non-numeric.")
            ctx_names.append(cname)
            ctx_cols.append(arr)
            continue
        if labels is not None:
            arr = _select_context_from_labels(meta, meta_upper, labels, cname)
            if arr is not None:
                ctx_names.append(cname)
                ctx_cols.append(arr)
                continue
        missing_cols.append(cname)

    if missing_cols:
        raise KeyError(f"Missing requested context columns: {missing_cols}")
    if len(ctx_cols) == 0:
        raise ValueError("No usable context columns were assembled.")
    ctx_mat = np.column_stack(ctx_cols).astype(np.float32)

    # Ensure one-to-one row alignment between coefficients and contexts.
    if coef_mat.shape[0] != ctx_mat.shape[0]:
        raise ValueError(
            f"Row count mismatch: COEF has {coef_mat.shape[0]} rows, META has {ctx_mat.shape[0]} rows"
        )

    # Keep only fully finite rows used by downstream training.
    good = np.isfinite(coef_mat).all(axis=1) & np.isfinite(ctx_mat).all(axis=1)
    coef_mat = coef_mat[good]
    ctx_mat = ctx_mat[good]
    return coef_mat, ctx_mat, coef_names, ctx_names

def _find_chi2_column(meta_tbl):
    """Find a reduced-chi^2-like column in decomposition META table.

    The decomposition outputs can vary by naming convention across runs; this
    helper probes common aliases in priority order.
    """
    names = {c.upper(): c for c in meta_tbl.colnames}
    for cand in ["REDUCED_CHI2", "CHI2_REDUCED", "CHI2", "RCHI2"]:
        if cand in names:
            return names[cand]
    raise KeyError("No chi2-like column found in decomposition META table")

def read_decomp_dataset_with_chi2(
    decomp_fits_path,
    input_fits_path,
    context_columns,
    decomp_kind="sky1",
):
    """Read decomposition dataset plus aligned reduced-chi^2 values.

    The returned `chi2_used` vector is masked with the *same* finite-row filter
    applied to coefficient/context matrices, guaranteeing strict row alignment.
    """
    coef_mat, ctx_mat, coef_names, ctx_names = read_decomp_dataset(
        decomp_fits_path=decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind=decomp_kind,
    )
    kind = decomp_kind.lower()

    # Reload raw tables to reproduce the exact finite-mask construction.
    with fits.open(decomp_fits_path) as hdul_dec:
        coef_tbl = _coerce_coef_hdu_to_table(hdul_dec["COEF"])
        dec_meta = Table(hdul_dec["META"].data)
    with fits.open(input_fits_path) as hdul_in:
        in_meta = Table(hdul_in["META"].data)

    coef_cols = []
    for name in list(coef_tbl.colnames):
        arr = _as_array(coef_tbl[name])
        if arr is not None:
            coef_cols.append(arr)
    coef_raw = np.column_stack(coef_cols).astype(np.float32)

    meta_upper = {c.upper(): c for c in in_meta.colnames}
    labels = None
    if kind in ("sky1", "sky2"):
        label_col = "SKY_NEAR_LABEL" if kind == "sky1" else "SKY_FAR_LABEL"
        labels = np.char.upper(np.char.strip(np.asarray(in_meta[meta_upper[label_col]]).astype(str)))

    ctx_cols = []
    for cname in context_columns:
        key = cname.upper()
        if key in meta_upper:
            arr = _as_array(in_meta[meta_upper[key]])
            if arr is None:
                raise ValueError(f"Context column '{cname}' is non-numeric.")
            ctx_cols.append(arr)
        else:
            if labels is None:
                raise KeyError(f"Missing context column: {cname}")
            arr = _select_context_from_labels(in_meta, meta_upper, labels, cname)
            if arr is None:
                raise KeyError(f"Missing context column: {cname}")
            ctx_cols.append(arr)
    ctx_raw = np.column_stack(ctx_cols).astype(np.float32)

    good = np.isfinite(coef_raw).all(axis=1) & np.isfinite(ctx_raw).all(axis=1)
    chi2_col = _find_chi2_column(dec_meta)
    chi2_full = np.asarray(dec_meta[chi2_col], dtype=np.float64)
    if chi2_full.shape[0] != good.shape[0]:
        raise ValueError(
            f"chi2 rows ({chi2_full.shape[0]}) do not match decomposition rows ({good.shape[0]})"
        )
    chi2_used = chi2_full[good]
    if chi2_used.shape[0] != coef_mat.shape[0]:
        raise ValueError(
            f"Aligned chi2 rows ({chi2_used.shape[0]}) do not match coef rows ({coef_mat.shape[0]})"
        )
    return coef_mat, ctx_mat, coef_names, ctx_names, chi2_used

class RobustScaler:
    """Median/IQR scaler with safe fallback for near-constant columns.

    This is intentionally robust to heavy-tailed coefficient distributions.
    """
    def fit(self, x):
        """Estimate per-column median and IQR scale."""
        self.med_ = np.nanmedian(x, axis=0)
        q25 = np.nanpercentile(x, 25, axis=0)
        q75 = np.nanpercentile(x, 75, axis=0)
        iqr = q75 - q25
        self.scale_ = np.where(iqr > 1e-8, iqr, 1.0)
        return self

    def transform(self, x):
        """Normalize features using fitted median and IQR."""
        return (x - self.med_) / self.scale_

    def inverse_transform(self, x):
        """Map normalized features back to original space."""
        return x * self.scale_ + self.med_

def _coef_to_model_space(coef):
    """Map physical non-negative coefficients to model space.

    The square-root transform reduces dynamic range and stabilizes optimization.
    """
    return np.sqrt(np.clip(coef, 0.0, None)).astype(np.float32)

def _coef_from_model_space(coef_model):
    """Inverse of `_coef_to_model_space` (clip + square)."""
    coef_model = np.clip(np.asarray(coef_model, dtype=np.float32), 0.0, None)
    return np.square(coef_model).astype(np.float32)

class ConditionalVAE(nn.Module):
    """Conditional VAE modeling coefficient vectors given context features."""

    def __init__(self, n_coef, n_ctx, z_dim=16, hidden=256):
        """Construct encoder/decoder MLPs and latent heads."""
        super().__init__()
        enc_in = n_coef + n_ctx
        dec_in = z_dim + n_ctx
        self.encoder = nn.Sequential(
            nn.Linear(enc_in, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
        )
        self.mu = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)
        self.decoder = nn.Sequential(
            nn.Linear(dec_in, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, n_coef),
        )

    def encode(self, coef, ctx):
        """Encode `(coef, ctx)` into latent Gaussian parameters `(mu, logvar)`."""
        h = self.encoder(torch.cat([coef, ctx], dim=-1))
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        """Sample latent vector with the reparameterization trick."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, ctx):
        """Decode latent state `z` conditioned on context `ctx`."""
        return self.decoder(torch.cat([z, ctx], dim=-1))

    def forward(self, coef, ctx):
        """Compute one stochastic VAE pass and return reconstruction + latent stats."""
        mu, logvar = self.encode(coef, ctx)
        z = self.reparameterize(mu, logvar)
        coef_hat = self.decode(z, ctx)
        return coef_hat, mu, logvar

def cvae_loss_stable(
    coef_hat,
    coef_true,
    mu,
    logvar,
    beta=1.0,
    latent_l2_weight=0.0,
    coef_weights=None,
    upper_bounds=None,
    upper_weight=0.0,
):
    """Stable CVAE objective with optional domain priors.

    Terms:
    - reconstruction: smooth-L1, optionally coefficient-weighted
    - KL divergence: computed from clamped log-variance for numerical stability
    - latent L2 penalty: weak pull on latent mean magnitude
    - upper-bound penalty: soft cap for selected coefficients (e.g. moon terms)
    """
    recon_raw = F.smooth_l1_loss(coef_hat, coef_true, reduction="none")
    if coef_weights is not None:
        weights = coef_weights.to(coef_true.device).view(1, -1)
        recon = (recon_raw * weights).sum() / (weights.sum() * coef_true.shape[0])
    else:
        recon = recon_raw.mean()

    latent_penalty = latent_l2_weight * mu.pow(2).mean() if latent_l2_weight > 0.0 else 0.0

    upper_penalty = 0.0
    if upper_bounds is not None and upper_weight > 0.0:
        bounds = upper_bounds.to(coef_hat.device).view(1, -1)
        upper_penalty = torch.square(F.relu(coef_hat - bounds)).mean()

    # Clamp logvar to avoid extreme exponentials in KL computation.
    logvar_c = torch.clamp(logvar, -12.0, 12.0)
    kl = -0.5 * torch.mean(1.0 + logvar_c - mu.pow(2) - logvar_c.exp())
    loss = recon + beta * kl + latent_penalty + upper_weight * upper_penalty
    return loss, recon.detach(), kl.detach()

def split_indices(n, train_frac=0.8, val_frac=0.1, seed=42):
    """Randomly split indices into train/validation/test subsets."""
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx

def make_loader(coef, ctx, idx, batch_size=64, shuffle=False, generator=None):
    """Create a PyTorch DataLoader from selected row indices.

    A fixed `generator` makes shuffled batches reproducible across reruns.
    """
    ds = TensorDataset(
        torch.from_numpy(coef[idx]).float(),
        torch.from_numpy(ctx[idx]).float(),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=generator)

def _set_reproducibility(seed=42, deterministic=True):
    """Seed Python/NumPy/Torch and optionally request deterministic kernels."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            # Some ops/backends may not support strict determinism.
            pass

def _finite_report(name, arr):
    """Print a quick finite-value and range summary for diagnostics."""
    arr = np.asarray(arr)
    fin = np.isfinite(arr)
    print(
        f"{name}: shape={arr.shape}, finite={fin.mean()*100:.2f}%, "
        f"min={np.nanmin(arr):.4g}, max={np.nanmax(arr):.4g}"
    )

def _per_dim_quantile_bounds(arr, q_low=0.5, q_high=99.5):
    """Return per-dimension quantile bounds with safe ordering."""
    lo = np.nanpercentile(arr, q_low, axis=0).astype(np.float32)
    hi = np.nanpercentile(arr, q_high, axis=0).astype(np.float32)
    hi = np.where(hi <= lo, lo + 1e-6, hi).astype(np.float32)
    return lo, hi

def train_cvae_stable(
    coef,
    ctx,
    coef_names=None,
    z_dim=16,
    hidden=256,
    lr=1e-3,
    batch_size=64,
    n_epochs=120,
    beta_max=0.8,
    beta_warmup_epochs=40,
    latent_l2_weight=1e-4,
    grad_clip=1.0,
    seed=42,
):
    """Train a numerically guarded conditional VAE and return artifacts.

    Workflow:
    1) Validate inputs and map coefficients to model space.
    2) Build robust scalers from training split only.
    3) Train with beta warmup, gradient clipping, and finite-loss guards.
    4) Track best validation checkpoint and evaluate on held-out test split.
    """
    _set_reproducibility(seed=seed, deterministic=True)

    if not np.isfinite(coef).all() or not np.isfinite(ctx).all():
        raise ValueError("Input coef/ctx contain non-finite values before training.")

    coef_model = _coef_to_model_space(coef)
    _finite_report("coef(raw)", coef)
    _finite_report("coef(model)", coef_model)
    _finite_report("ctx(raw)", ctx)

    # Optional priors: upweight specific coefficient families and cap moon tails.
    coef_weights_t = None
    upper_bounds_t = None
    if coef_names is not None:
        coef_weights = np.ones(coef.shape[1], dtype=np.float32)
        upper_bounds = np.full(coef.shape[1], np.inf, dtype=np.float32)
        boosted_positive = []
        boosted_moon = []
        for i, name in enumerate(coef_names):
            lname = str(name).lower()
            if lname.startswith("o2") or lname.startswith("feo") or lname.startswith("ho2"):
                coef_weights[i] = 2.0
                boosted_positive.append(str(name))
            elif lname.startswith("moon_bs"):
                coef_weights[i] = 1.5
                upper_bounds[i] = np.float32(np.nanpercentile(coef_model[:, i], 99.5) * 1.05)
                boosted_moon.append(str(name))
        print(f"Positive-component weighted coefficients: {len(boosted_positive)} boosted")
        if boosted_positive:
            print("  " + ", ".join(boosted_positive))
        print(f"Moon_bs-weighted coefficients: {len(boosted_moon)} boosted")
        coef_weights_t = torch.from_numpy(coef_weights)
        upper_bounds_t = torch.from_numpy(upper_bounds)

    train_idx, val_idx, test_idx = split_indices(len(coef), seed=seed)

    # Fit scalers on train split only to avoid leakage.
    coef_scaler = RobustScaler().fit(coef_model[train_idx])
    ctx_scaler = RobustScaler().fit(ctx[train_idx])
    coef_n = coef_scaler.transform(coef_model).astype(np.float32)
    ctx_n = ctx_scaler.transform(ctx).astype(np.float32)

    # Last-resort clipping for extreme normalized tails.
    coef_n = np.clip(coef_n, -25.0, 25.0)
    ctx_n = np.clip(ctx_n, -25.0, 25.0)
    if not np.isfinite(coef_n).all() or not np.isfinite(ctx_n).all():
        raise ValueError("Non-finite values after scaling/clipping.")

    _finite_report("coef(norm)", coef_n)
    _finite_report("ctx(norm)", ctx_n)

    # Fixed shuffling order across reruns.
    loader_rng = torch.Generator()
    loader_rng.manual_seed(seed)
    tr_loader = make_loader(coef_n, ctx_n, train_idx, batch_size=batch_size, shuffle=True, generator=loader_rng)
    va_loader = make_loader(coef_n, ctx_n, val_idx, batch_size=batch_size, shuffle=False)
    te_loader = make_loader(coef_n, ctx_n, test_idx, batch_size=batch_size, shuffle=False)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ConditionalVAE(n_coef=coef.shape[1], n_ctx=ctx.shape[1], z_dim=z_dim, hidden=hidden).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    best = {"val_loss": np.inf, "state": None, "epoch": -1}
    history = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        beta = beta_max * min(1.0, epoch / max(beta_warmup_epochs, 1))

        tr_loss, tr_recon, tr_kl, n_batches, n_skipped = 0.0, 0.0, 0.0, 0, 0
        for coef_b, ctx_b in tr_loader:
            if not torch.isfinite(coef_b).all() or not torch.isfinite(ctx_b).all():
                n_skipped += 1
                continue

            coef_b = coef_b.to(device)
            ctx_b = ctx_b.to(device)
            opt.zero_grad(set_to_none=True)

            coef_hat, mu, logvar = model(coef_b, ctx_b)
            loss, recon, kl = cvae_loss_stable(
                coef_hat,
                coef_b,
                mu,
                logvar,
                beta=beta,
                latent_l2_weight=latent_l2_weight,
                coef_weights=coef_weights_t,
                upper_bounds=upper_bounds_t,
                upper_weight=0.05,
            )
            if not torch.isfinite(loss):
                n_skipped += 1
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            tr_loss += float(loss.item())
            tr_recon += float(recon.item())
            tr_kl += float(kl.item())
            n_batches += 1

        if n_batches == 0:
            raise RuntimeError(
                f"All training batches were skipped at epoch {epoch}. "
                "Check input scaling/context columns."
            )

        train_metrics = {
            "loss": tr_loss / n_batches,
            "recon": tr_recon / n_batches,
            "kl": tr_kl / n_batches,
            "n_skipped": n_skipped,
        }

        # Validation pass without gradients.
        model.eval()
        val_loss, val_recon, val_kl, n_val_batches = 0.0, 0.0, 0.0, 0
        with torch.no_grad():
            for coef_b, ctx_b in va_loader:
                coef_b = coef_b.to(device)
                ctx_b = ctx_b.to(device)
                coef_hat, mu, logvar = model(coef_b, ctx_b)
                loss, recon, kl = cvae_loss_stable(
                    coef_hat,
                    coef_b,
                    mu,
                    logvar,
                    beta=beta,
                    latent_l2_weight=latent_l2_weight,
                    coef_weights=coef_weights_t,
                    upper_bounds=upper_bounds_t,
                    upper_weight=0.05,
                )
                if not torch.isfinite(loss):
                    continue
                val_loss += float(loss.item())
                val_recon += float(recon.item())
                val_kl += float(kl.item())
                n_val_batches += 1

        if n_val_batches == 0:
            val_metrics = {"loss": np.inf, "recon": np.inf, "kl": np.inf}
        else:
            val_metrics = {
                "loss": val_loss / n_val_batches,
                "recon": val_recon / n_val_batches,
                "kl": val_kl / n_val_batches,
            }

        row = {
            "epoch": epoch,
            "beta": beta,
            "train_loss": train_metrics["loss"],
            "train_recon": train_metrics["recon"],
            "train_kl": train_metrics["kl"],
            "train_skipped": train_metrics["n_skipped"],
            "val_loss": val_metrics["loss"],
            "val_recon": val_metrics["recon"],
            "val_kl": val_metrics["kl"],
        }
        history.append(row)

        # Keep a CPU copy of the best validation checkpoint.
        if val_metrics["loss"] < best["val_loss"]:
            best["val_loss"] = val_metrics["loss"]
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best["epoch"] = epoch

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:03d} beta={beta:.3f} "
                f"train={train_metrics['loss']:.4f} val={val_metrics['loss']:.4f} "
                f"skipped={train_metrics['n_skipped']}"
            )

    if best["state"] is not None:
        model.load_state_dict(best["state"])

    # Final test evaluation at full beta weight.
    model.eval()
    te_loss, te_recon, te_kl, n_te_batches = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for coef_b, ctx_b in te_loader:
            coef_b = coef_b.to(device)
            ctx_b = ctx_b.to(device)
            coef_hat, mu, logvar = model(coef_b, ctx_b)
            loss, recon, kl = cvae_loss_stable(
                coef_hat,
                coef_b,
                mu,
                logvar,
                beta=beta_max,
                latent_l2_weight=latent_l2_weight,
                coef_weights=coef_weights_t,
                upper_bounds=upper_bounds_t,
                upper_weight=0.05,
            )
            if not torch.isfinite(loss):
                continue
            te_loss += float(loss.item())
            te_recon += float(recon.item())
            te_kl += float(kl.item())
            n_te_batches += 1

    if n_te_batches > 0:
        test_metrics = {
            "loss": te_loss / n_te_batches,
            "recon": te_recon / n_te_batches,
            "kl": te_kl / n_te_batches,
        }
    else:
        test_metrics = {"loss": np.inf, "recon": np.inf, "kl": np.inf}

    print(f"Best epoch: {best['epoch']} | best val loss: {best['val_loss']:.4f}")
    print(
        "Test metrics: "
        f"loss={test_metrics['loss']:.4f} recon={test_metrics['recon']:.4f} kl={test_metrics['kl']:.4f}"
    )

    artifacts = {
        "model": model,
        "coef_scaler": coef_scaler,
        "ctx_scaler": ctx_scaler,
        "history": history,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "device": device,
    }
    return artifacts

# ---- Run v1 stable on joined sky1 + sky2 decomposition files ----
# This doubles the effective input size by stacking sky1 and sky2 rows.
input_file = "lvmsframe_median_stack_1.2.1_every10.fits"
decomp_specs = [
    ("sky1", "lvmsframe_median_stack_1.2.1_every10_decomp_sky1.fits"),
    ("sky2", "lvmsframe_median_stack_1.2.1_every10_decomp_sky2.fits"),
]
context_cols = [
    "alt",
    "moon_sep",
    "moon_alt",
    "sun_alt",
    # "moon_phase",
    # "moon_fli",
    "moon_illum",
    "airmass"
]
joined_coef = []
joined_ctx = []
joined_chi2 = []
coef_names_ref = None
ctx_names_ref = None
for kind, decomp_path in decomp_specs:
    coef_k, ctx_k, coef_names_k, ctx_names_k, chi2_k = read_decomp_dataset_with_chi2(
        decomp_fits_path=decomp_path,
        input_fits_path=input_file,
        context_columns=context_cols,
        decomp_kind=kind,
    )
    if coef_names_ref is None:
        coef_names_ref = coef_names_k
        ctx_names_ref = ctx_names_k
    else:
        if coef_names_k != coef_names_ref:
            raise ValueError(f"Coefficient names mismatch for {kind}")
        if ctx_names_k != ctx_names_ref:
            raise ValueError(f"Context names mismatch for {kind}")
    joined_coef.append(coef_k)
    joined_ctx.append(ctx_k)
    joined_chi2.append(chi2_k)
coef_all = np.vstack(joined_coef)
ctx_all = np.vstack(joined_ctx)
chi2_used = np.concatenate(joined_chi2)
coef_names = coef_names_ref
ctx_names = ctx_names_ref

# ---- Chi2 filter controls ----
CHI2_MIN = 0.0
CHI2_MAX = 10.0        # Set None to use percentile cap only
CHI2_QMAX = 90.0       # Keep rows <= this percentile of reduced chi2
chi2_hi = np.nanpercentile(chi2_used[np.isfinite(chi2_used)], CHI2_QMAX)
chi2_upper = chi2_hi if CHI2_MAX is None else min(CHI2_MAX, chi2_hi)
mask = (
    np.isfinite(chi2_used)
    & (chi2_used >= CHI2_MIN)
    & (chi2_used <= chi2_upper)
)
print(
    f"Joined sky1+sky2 chi2 filter: min={CHI2_MIN:.3g}, qmax={CHI2_QMAX:.1f}%=>{chi2_hi:.3g}, "
    f"upper={chi2_upper:.3g} | keep={mask.sum()}/{len(mask)} "
    f"({100.0*mask.mean():.1f}%)"
)
fig_chi2_joined = px.histogram(
    x=chi2_used[mask],
    nbins=80,
    title="Joined sky1+sky2 reduced chi2 distribution (rows used for training)",
    labels={"x": "reduced chi2", "y": "count"},
)
fig_chi2_joined.update_layout(template="plotly_white", bargap=0.03)
fig_chi2_joined.show()
coef_mat = coef_all[mask]
ctx_mat = ctx_all[mask]
chi2_filtered = chi2_used[mask]

# Hard manual coefficient bounds for known problematic tails.
# Rows outside these ranges are dropped (not winsorized).
HARD_COEF_BOUNDS = {
    "feo": (0.0, 0.01),
    "atom_k": (0.0, 0.01),
}
coef_name_l = [str(n).lower() for n in coef_names]
manual_keep = np.ones(coef_mat.shape[0], dtype=bool)
for cname, (lo, hi) in HARD_COEF_BOUNDS.items():
    idxs = np.where(np.array(coef_name_l) == cname)[0]
    if idxs.size == 0:
        print(f"Manual hard clip: coefficient {cname} not found; skipping.")
        continue
    vals = coef_mat[:, idxs[0]]
    within = np.isfinite(vals) & (vals >= lo) & (vals <= hi)
    manual_keep &= within
    print(
        f"Manual hard clip {cname}: [{lo:.3g}, {hi:.3g}] | kept {within.sum()}/{within.size} "
        f"({100.0 * within.mean():.1f}%)"
    )
n_manual0 = coef_mat.shape[0]
coef_mat = coef_mat[manual_keep]
ctx_mat = ctx_mat[manual_keep]
chi2_filtered = chi2_filtered[manual_keep]
print(
    f"Manual hard clip combined: kept {manual_keep.sum()}/{n_manual0} "
    f"({100.0 * manual_keep.mean():.1f}%)"
)

# Second-stage outlier filtering after chi2: iterative kappa-sigma clipping (kappa=5).
def _kappa_sigma_row_mask(x, kappa=5.0, n_iter=3):
    """Return row mask via iterative per-feature kappa-sigma clipping."""
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x).all(axis=1)
    if not np.any(keep):
        return keep
    for _ in range(n_iter):
        mu = np.nanmean(x[keep], axis=0)
        sig = np.nanstd(x[keep], axis=0)
        sig = np.where(np.isfinite(sig) & (sig > 0), sig, 1.0)
        within = np.all(np.abs(x - mu) <= (kappa * sig), axis=1)
        within &= np.isfinite(x).all(axis=1)
        new_keep = keep & within
        if new_keep.sum() == keep.sum() or new_keep.sum() == 0:
            break
        keep = new_keep
    return keep

KAPPA = 6.0
mask_coef = _kappa_sigma_row_mask(coef_mat, kappa=KAPPA, n_iter=3)
mask_kappa = mask_coef
n0 = coef_mat.shape[0]
coef_mat = coef_mat[mask_kappa]
ctx_mat = ctx_mat[mask_kappa]
chi2_filtered = chi2_filtered[mask_kappa]
print(
    f"Kappa-sigma filter (kappa={KAPPA:.1f}): kept {mask_kappa.sum()}/{n0} "
    f"({100.0 * mask_kappa.mean():.1f}%)"
)
print(f"Post-kappa shapes: coef_mat={coef_mat.shape}, ctx_mat={ctx_mat.shape}")


# Multipanel histogram of selected filtered coefficients:
# moon spline median + non-OH coefficients (excluding individual moon_bs terms).
coef_name_l = [str(n).lower() for n in coef_names]
moon_bs_idx = np.array([i for i, n in enumerate(coef_name_l) if n.startswith("moon_bs")], dtype=int)
hist_frames = []

if moon_bs_idx.size > 0:
    moon_bs_median = np.nanmedian(coef_mat[:, moon_bs_idx], axis=1)
    moon_bs_median = moon_bs_median[np.isfinite(moon_bs_median)]
    if moon_bs_median.size > 0:
        hist_frames.append(pandas.DataFrame({"coefficient": "moon_bs_median", "value": moon_bs_median.astype(np.float64)}))

for i, name in enumerate(coef_names):
    lname = str(name).lower()
    if "oh" in lname or lname.startswith("moon_bs"):
        continue
    vals = coef_mat[:, i]
    print(lname, np.min(vals), np.max(vals), np.mean(vals), np.median(vals), np.std(vals))
    #vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        continue
    hist_frames.append(pandas.DataFrame({"coefficient": str(name), "value": vals.astype(np.float64)}))

if len(hist_frames) > 0:
    hist_df = pandas.concat(hist_frames, ignore_index=True)
    n_panels = int(hist_df["coefficient"].nunique())
    n_cols = min(12, max(4, int(np.ceil(np.sqrt(n_panels * 1.5)))))
    n_rows = int(np.ceil(n_panels / n_cols))
    row_spacing = min(0.2, 0.9 / max(n_rows - 1, 1))
    col_spacing = min(0.02, 0.9 / max(n_cols - 1, 1))
    print(f"Histogram facets: n_panels={n_panels}, n_cols={n_cols}, n_rows={n_rows}")
    fig_coef_hist = px.histogram(
        hist_df,
        x="value",
        facet_col="coefficient",
        facet_col_wrap=n_cols,
        facet_row_spacing=row_spacing,
        facet_col_spacing=col_spacing,
        nbins=70,
        title="Filtered coefficient distributions (moon_bs_median + non-OH coefficients)",
        labels={"value": "coefficient value", "count": "count"},
    )
    fig_coef_hist.for_each_annotation(lambda a: a.update(text=a.text.replace("coefficient=", "")))
    # Keep per-facet bins independent.
    for i, tr in enumerate(fig_coef_hist.data):
        tr.update(bingroup=f"facet_{i}", autobinx=True)
    # Force scientific tick labels and visible x ticks on all facets.
    fig_coef_hist.update_xaxes(
        matches=None,
        showticklabels=True,
        ticks="outside",
        ticklen=4,
        exponentformat="e",
        showexponent="all",
        tickformat=".2e",
    )
    fig_coef_hist.update_yaxes(
        matches=None,
        exponentformat="e",
        showexponent="all",
        tickformat=".2e",
    )
    fig_coef_hist.update_layout(
        template="plotly_white",
        bargap=0.02,
        height=280 * n_rows,
    )
    # Add per-panel vertical guides: median and median + 3*std.
    shapes = []
    for tr in fig_coef_hist.data:
        x = np.asarray(tr.x, dtype=np.float64)
        x = x[np.isfinite(x)]
        if x.size == 0:
            continue
        x_med = float(np.nanmedian(x))
        x_p3s = float(x_med + 3.0 * np.nanstd(x))
        xref = tr.xaxis if tr.xaxis else "x"
        yaxis_id = tr.yaxis if tr.yaxis else "y"
        yref = f"{yaxis_id} domain"
        shapes.append(
            dict(
                type="line",
                x0=x_med,
                x1=x_med,
                y0=0.0,
                y1=1.0,
                xref=xref,
                yref=yref,
                line=dict(color="#1b9e77", width=1.4, dash="dash"),
            )
        )
        shapes.append(
            dict(
                type="line",
                x0=x_p3s,
                x1=x_p3s,
                y0=0.0,
                y1=1.0,
                xref=xref,
                yref=yref,
                line=dict(color="#d95f02", width=1.4, dash="dot"),
            )
        )
    fig_coef_hist.update_layout(shapes=shapes)
    fig_coef_hist.show()
else:
    print("No coefficients available for histogram plotting after filtering.")
print(f"Loaded {coef_mat.shape[0]} filtered samples | n_coef={coef_mat.shape[1]} | n_ctx={ctx_mat.shape[1]}")
print("Context columns:", ctx_names)
#raise RuntimeError("Stop here for inspection before training CVAE. Set to False to continue.")
artifacts = train_cvae_stable(
    coef_mat,
    ctx_mat,
    coef_names=coef_names,
    z_dim=16,
    hidden=256,
    lr=1e-3,
    batch_size=64,
    n_epochs=200,
    beta_max=1.0,
    beta_warmup_epochs=50,
    latent_l2_weight=1e-4,
    grad_clip=1.0,
)
model = artifacts["model"]
model.eval()
device = artifacts["device"]
train_idx = artifacts["train_idx"]
test_idx = artifacts["test_idx"]
coef_scaler = artifacts["coef_scaler"]
ctx_scaler = artifacts["ctx_scaler"]

coef_train_model = _coef_to_model_space(coef_mat[train_idx])
coef_test_model = _coef_to_model_space(coef_mat[test_idx])
coef_train = coef_scaler.transform(coef_train_model).astype(np.float32)
coef_test = coef_scaler.transform(coef_test_model).astype(np.float32)
ctx_train = ctx_scaler.transform(ctx_mat[train_idx]).astype(np.float32)
ctx_test = ctx_scaler.transform(ctx_mat[test_idx]).astype(np.float32)

# Train-set quantile bounds to suppress latent and output outliers during inference.
z_q_low, z_q_high = 0.5, 99.5
coef_q_low, coef_q_high = 0.5, 99.5
with torch.no_grad():
    coef_tr_t = torch.from_numpy(coef_train).to(device)
    ctx_tr_t = torch.from_numpy(ctx_train).to(device)
    mu_tr, _ = model.encode(coef_tr_t, ctx_tr_t)
z_lo, z_hi = _per_dim_quantile_bounds(mu_tr.cpu().numpy(), q_low=z_q_low, q_high=z_q_high)
coef_lo, coef_hi = _per_dim_quantile_bounds(coef_train, q_low=coef_q_low, q_high=coef_q_high)
print(
    f"Outlier-control bounds (Cell 7): latent q=[{z_q_low:.1f}, {z_q_high:.1f}], "
    f"coef_n q=[{coef_q_low:.1f}, {coef_q_high:.1f}]"
)

with torch.no_grad():
    coef_t = torch.from_numpy(coef_test).to(device)
    ctx_t = torch.from_numpy(ctx_test).to(device)
    # Deterministic prediction: decode from encoder mean instead of sampling z.
    mu_t, _ = model.encode(coef_t, ctx_t)

    # Clip latent coordinates to training-domain quantiles.
    z_lo_t = torch.from_numpy(z_lo).to(device)
    z_hi_t = torch.from_numpy(z_hi).to(device)
    mu_t_clip = torch.minimum(torch.maximum(mu_t, z_lo_t), z_hi_t)

    coef_hat_t_raw = model.decode(mu_t_clip, ctx_t)

    # Clip decoded normalized outputs to training-domain quantiles.
    coef_lo_t = torch.from_numpy(coef_lo).to(device)
    coef_hi_t = torch.from_numpy(coef_hi).to(device)
    coef_hat_t = torch.minimum(torch.maximum(coef_hat_t_raw, coef_lo_t), coef_hi_t)

z_clip_frac = float((mu_t_clip != mu_t).float().mean().item())
coef_clip_frac = float((coef_hat_t != coef_hat_t_raw).float().mean().item())
print(f"Cell 7 latent clipping fraction: {100.0 * z_clip_frac:.2f}%")
print(f"Cell 7 decoded-output clipping fraction: {100.0 * coef_clip_frac:.2f}%")

coef_hat_model = coef_scaler.inverse_transform(coef_hat_t.cpu().numpy())
coef_hat_phys = _coef_from_model_space(coef_hat_model)
print("Reconstruction array shape:", coef_hat_phys.shape)


Joined sky1+sky2 chi2 filter: min=0, qmax=90.0%=>4.68, upper=4.68 | keep=3106/3452 (90.0%)


Manual hard clip feo: [0, 0.01] | kept 2531/3106 (81.5%)
Manual hard clip atom_k: [0, 0.01] | kept 2460/3106 (79.2%)
Manual hard clip combined: kept 2089/3106 (67.3%)
Kappa-sigma filter (kappa=6.0): kept 1250/2089 (59.8%)
Post-kappa shapes: coef_mat=(1250, 442), ctx_mat=(1250, 6)
ho2 6.1568826e-11 1.3662086e-08 1.9164093e-09 1.1731716e-09 2.0427582e-09
feo 7.345118e-11 9.8462465e-08 2.92027e-09 1.5959984e-09 5.3105715e-09
o2ac 1.1890099e-10 1.0475093 0.29096484 0.2533835 0.14832945
atom_k 1.3063234e-07 0.00048653825 1.4891435e-05 5.5132514e-06 3.7757945e-05
atom_n 0.36575598 32.01714 6.903464 5.1740847 5.225614
atom_na 6.2287807 119.477486 42.195507 37.789635 21.966473
atom_og 60.213135 757.7349 291.38806 279.4555 111.56662
atom_or 13.194944 1282.8423 227.79773 156.42926 200.56564
atom_orc_oi0777 1.2165244e-09 9.359874 0.6203632 1.3714259e-07 1.279714
atom_orc_oi0845 1.0141065e-07 11.4488125 1.4531621 0.9428818 1.6115217
o2_b01 2.594085 3.0419192 2.90352 2.91917 0.06876963
Histogram fa

Loaded 1250 filtered samples | n_coef=442 | n_ctx=6
Context columns: ['alt', 'moon_sep', 'moon_alt', 'sun_alt', 'moon_illum', 'airmass']
coef(raw): shape=(1250, 442), finite=100.00%, min=1.606e-11, max=4.539e+05
coef(model): shape=(1250, 442), finite=100.00%, min=4.008e-06, max=673.7
ctx(raw): shape=(1250, 6), finite=100.00%, min=-84.29, max=174.4
Positive-component weighted coefficients: 4 boosted
  HO2, FeO, O2Ac, O2_b01
Moon_bs-weighted coefficients: 29 boosted
coef(norm): shape=(1250, 442), finite=100.00%, min=-4.41, max=25
ctx(norm): shape=(1250, 6), finite=100.00%, min=-9.027, max=25
epoch=001 beta=0.020 train=0.2921 val=0.2499 skipped=0
epoch=010 beta=0.200 train=0.1935 val=0.2066 skipped=0
epoch=020 beta=0.400 train=0.2131 val=0.2248 skipped=0
epoch=030 beta=0.600 train=0.2251 val=0.2388 skipped=0
epoch=040 beta=0.800 train=0.2383 val=0.2554 skipped=0
epoch=050 beta=1.000 train=0.2441 val=0.2455 skipped=0
epoch=060 beta=1.000 train=0.2408 val=0.2602 skipped=0
epoch=070 beta=1.0

In [5]:
# Diagnostics: coefficient-wise metrics + latent diagnostics
import numpy as np
import pandas as pd
from astropy.table import Table
import plotly.express as px

required = [
    "artifacts", "coef_mat", "ctx_mat", "coef_names",
    "coef_hat_phys", "mu_t_clip"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Cell 7 first. Missing variables: " + ", ".join(missing)
    )

model = artifacts["model"]
model.eval()

device = artifacts["device"]
test_idx = artifacts["test_idx"]
coef_scaler = artifacts["coef_scaler"]
ctx_scaler = artifacts["ctx_scaler"]

# Use training-time context names when available; avoid global overwrite from later cells.
ctx_names_train = artifacts.get("ctx_names", None)
if ctx_names_train is None:
    if "context_cols" in globals() and len(context_cols) == ctx_mat.shape[1]:
        ctx_names_train = list(context_cols)
    elif "ctx_names" in globals() and len(ctx_names) == ctx_mat.shape[1]:
        ctx_names_train = list(ctx_names)
    else:
        ctx_names_train = [f"ctx_{i}" for i in range(ctx_mat.shape[1])]

coef_test_phys = coef_mat[test_idx].astype(np.float32)
ctx_test_phys = ctx_mat[test_idx].astype(np.float32)

# Consume Cell 7 outputs directly (no duplicate clipping/re-encoding here).
if coef_hat_phys.shape != coef_test_phys.shape:
    raise RuntimeError(
        f"Prediction shape mismatch. coef_hat_phys={coef_hat_phys.shape}, coef_test_phys={coef_test_phys.shape}. "
        "Re-run Cell 7 before diagnostics."
    )

if "z_clip_frac" in globals() and "coef_clip_frac" in globals():
    print(f"Latent clipping fraction (from Cell 7): {100.0 * float(z_clip_frac):.2f}%")
    print(f"Decoded-output clipping fraction (from Cell 7): {100.0 * float(coef_clip_frac):.2f}%")

# ----- Coefficient-wise metrics -----
rmse = np.sqrt(np.mean((coef_hat_phys - coef_test_phys) ** 2, axis=0))
mae = np.mean(np.abs(coef_hat_phys - coef_test_phys), axis=0)

corr = []
for j in range(coef_test_phys.shape[1]):
    x = coef_test_phys[:, j]
    y = coef_hat_phys[:, j]
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        corr.append(np.nan)
    else:
        corr.append(float(np.corrcoef(x, y)[0, 1]))
corr = np.array(corr)

metrics_tbl = Table(
    {
        "coef_name": coef_names,
        "rmse": rmse,
        "mae": mae,
        "corr": corr,
    }
)
metrics_tbl.sort("rmse")

print("Top 15 coefficients by lowest RMSE:")
metrics_tbl[:15]

print("\nWorst 15 coefficients by RMSE:")
metrics_tbl[::-1][:15]

print("\nGlobal summary on test set:")
print(f"  mean RMSE = {np.nanmean(rmse):.5g}")
print(f"  median RMSE = {np.nanmedian(rmse):.5g}")
print(f"  mean corr = {np.nanmean(corr):.5g}")
print(f"  median corr = {np.nanmedian(corr):.5g}")

o2_mask = np.array([str(name).lower().startswith("o2") for name in coef_names])
if np.any(o2_mask):
    print("\nO2-specific summary:")
    print(f"  mean RMSE = {np.nanmean(rmse[o2_mask]):.5g}")
    print(f"  median RMSE = {np.nanmedian(rmse[o2_mask]):.5g}")
    print(f"  mean corr = {np.nanmean(corr[o2_mask]):.5g}")
    print(f"  median corr = {np.nanmedian(corr[o2_mask]):.5g}")

    o2_tbl = Table(
        {
            "coef_name": np.array(coef_names)[o2_mask],
            "rmse": rmse[o2_mask],
            "mae": mae[o2_mask],
            "corr": corr[o2_mask],
        }
    )
    o2_tbl.sort("rmse")
    print("\nO2 coefficient details:")
    o2_tbl

# ----- Robust scaling helpers for plotting -----
def _robust_bounds(arr, q_low=1.0, q_high=99.0, pad_frac=0.05):
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (-1.0, 1.0)

    lo, hi = np.nanpercentile(arr, [q_low, q_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = np.nanmin(arr)
        hi = np.nanmax(arr)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        c = float(arr[0])
        lo, hi = c - 0.5, c + 0.5

    span = max(hi - lo, 1e-6)
    pad = pad_frac * span
    return lo - pad, hi + pad

def _clip_to_bounds(arr, bounds):
    return np.clip(np.asarray(arr, dtype=np.float64), bounds[0], bounds[1])

def _set_bottom_facet_x_titles(fig, col_names, n_rows):
    """Show per-facet x-axis titles only on the bottom row."""
    n_cols = len(col_names)
    for r in range(n_rows):
        for c, cname in enumerate(col_names):
            axis_idx = r * n_cols + c + 1
            axis_name = "xaxis" if axis_idx == 1 else f"xaxis{axis_idx}"
            if axis_name in fig.layout:
                # Plotly numbers facet axes from bottom to top.
                fig.layout[axis_name].title.text = cname if r == 0 else ""

def _set_left_facet_y_titles(fig, row_titles, n_cols):
    """Set left y-axis titles per row to match output-group row labels."""
    # Plotly numbers y-axes from bottom to top across facet rows.
    ordered = list(row_titles)[::-1]
    for r, title in enumerate(ordered):
        axis_idx = r * n_cols + 1
        axis_name = "yaxis" if axis_idx == 1 else f"yaxis{axis_idx}"
        if axis_name in fig.layout:
            fig.layout[axis_name].title.text = title

# ----- Latent diagnostics -----
# Use clipped latent coordinates produced in Cell 7.
mu_np = mu_t_clip.cpu().numpy()

# Compact grid: columns are context variables and rows are latent dims.
max_latent_dims = min(4, mu_np.shape[1])
latent_cols = [f"z{k}" for k in range(max_latent_dims)]

# Downsample for responsive plotting on large test sets.
max_points = 5000
n_test = mu_np.shape[0]
if n_test > max_points:
    rng = np.random.default_rng(42)
    keep = np.sort(rng.choice(n_test, size=max_points, replace=False))
else:
    keep = np.arange(n_test)

mu_plot = mu_np[keep, :max_latent_dims]
ctx_plot = ctx_test_phys[keep]

rows = []
for k, zname in enumerate(latent_cols):
    for j, cname in enumerate(ctx_names_train):
        x_bounds = _robust_bounds(ctx_plot[:, j])
        y_bounds = _robust_bounds(mu_plot[:, k])
        rows.append(
            pd.DataFrame(
                {
                    "context_param": cname,
                    "context_value": _clip_to_bounds(ctx_plot[:, j], x_bounds),
                    "latent_value": _clip_to_bounds(mu_plot[:, k], y_bounds),
                    "latent_dim": zname,
                }
            )
        )

latent_ctx_df = pd.concat(rows, ignore_index=True)

fig_ctx = px.scatter(
    latent_ctx_df,
    x="context_value",
    y="latent_value",
    facet_col="context_param",
    facet_row="latent_dim",
    opacity=0.28,
    render_mode="webgl",
    title="Latent vs context grid (robust axis scaling)",
    labels={"latent_value": "latent value", "context_value": ""},
)

# Replace generic facet annotation text so each column/row clearly shows variable names.
fig_ctx.for_each_annotation(
    lambda a: a.update(
        text=a.text.replace("context_param=", "").replace("latent_dim=", "")
    )
)
fig_ctx.update_xaxes(matches=None)
fig_ctx.update_yaxes(matches=None)
fig_ctx.update_layout(
    template="plotly_white",
    height=max(650, 220 * max_latent_dims),
)
_set_bottom_facet_x_titles(fig_ctx, ctx_names_train, max_latent_dims)
fig_ctx.show()

# ----- Targeted output-vs-context grid -----
# Row 1: median of Moon_bs coefficients.
# Following rows: one continuum component each, excluding OH groups.
coef_name_l = [str(n).lower() for n in coef_names]

moon_bs_idx = np.array(
    [i for i, n in enumerate(coef_name_l) if n.startswith("moon_bs")],
    dtype=int,
)

def _continuum_group(name):
    # Explicitly skip OH-related groups/components.
    if "oh" in name:
        return None

    patterns = [
        ("moon", ["moon", "zodi"]),
        ("diffuse", ["diffuse"]),
        ("ho2", ["ho2", "hydroperoxyl"]),
        ("feo", ["feo", "iron"]),
        ("o2", ["o2", "oxygen"]),
        ("continuum", ["continuum"]),
    ]
    for gname, keys in patterns:
        if any(k in name for k in keys):
            return gname
    return None

comp_to_idx = {}
for i, n in enumerate(coef_name_l):
    if i in moon_bs_idx:
        continue
    g = _continuum_group(n)
    if g is None:
        continue
    comp_to_idx.setdefault(g, []).append(i)

row_groups = []
if moon_bs_idx.size > 0:
    row_groups.append(("moon_bs_median", moon_bs_idx))

for gname in sorted(comp_to_idx.keys()):
    row_groups.append((gname, np.array(comp_to_idx[gname], dtype=int)))

if len(row_groups) == 0:
    raise RuntimeError(
        "Could not identify Moon_bs/continuum groups from coefficient names. "
        "Inspect coef_names naming conventions."
    )

out_rows = []
for row_name, idxs in row_groups:
    y_true = np.nanmedian(coef_test_phys[:, idxs], axis=1)
    y_pred = np.nanmedian(coef_hat_phys[:, idxs], axis=1)

    for j, cname in enumerate(ctx_names_train):
        x_raw = ctx_test_phys[:, j]
        x_bounds = _robust_bounds(x_raw)
        y_bounds = _robust_bounds(np.concatenate([y_true, y_pred]))

        out_rows.append(
            pd.DataFrame(
                {
                    "output_group": row_name,
                    "context_param": cname,
                    "context_value": _clip_to_bounds(x_raw, x_bounds),
                    "output_value": _clip_to_bounds(y_true, y_bounds),
                    "series": "true",
                }
            )
        )
        out_rows.append(
            pd.DataFrame(
                {
                    "output_group": row_name,
                    "context_param": cname,
                    "context_value": _clip_to_bounds(x_raw, x_bounds),
                    "output_value": _clip_to_bounds(y_pred, y_bounds),
                    "series": "learned",
                }
            )
        )

out_df = pd.concat(out_rows, ignore_index=True)

fig_out = px.scatter(
    out_df,
    x="context_value",
    y="output_value",
    color="series",
    facet_col="context_param",
    facet_row="output_group",
    opacity=0.30,
    render_mode="webgl",
    title="Targeted outputs vs context (Moon_bs median + continuum components, no OH)",
    color_discrete_map={"true": "#1f77b4", "learned": "#d62728"},
    labels={"output_value": "", "context_value": ""},
)
fig_out.for_each_annotation(
    lambda a: a.update(
        text=a.text.replace("context_param=", "").replace("output_group=", "")
    )
)
fig_out.update_xaxes(matches=None)
fig_out.update_yaxes(matches=None)
fig_out.update_layout(
    template="plotly_white",
    height=max(720, 220 * len(row_groups)),
)
_set_bottom_facet_x_titles(fig_out, ctx_names_train, len(row_groups))
row_titles = [name for name, _ in row_groups]
_set_left_facet_y_titles(fig_out, row_titles, len(ctx_names_train))
fig_out.show()

Latent clipping fraction (from Cell 7): 1.65%
Decoded-output clipping fraction (from Cell 7): 9.98%
Top 15 coefficients by lowest RMSE:

Worst 15 coefficients by RMSE:

Global summary on test set:
  mean RMSE = 188.2
  median RMSE = 0.016485
  mean corr = 0.63378
  median corr = 0.7025

O2-specific summary:
  mean RMSE = 0.11275
  median RMSE = 0.11275
  mean corr = 0.49943
  median corr = 0.49943

O2 coefficient details:
